In [7]:
# step 1 견인 데이터 병합

import pandas as pd
import os

# 1. 경로 설정
path = r'C:\Users\chpar\Exercise_CP\work'

# 2. 데이터 병합을 위한 리스트
towing_list = []

# 3. 1월부터 12월까지 순회 (range(1, 13)은 1부터 12까지입니다!)
for i in range(1, 13):
    file_name = f'서울특별시_전동킥보드_견인_현황_{i}월.csv'
    full_path = os.path.join(path, file_name)
    
    if os.path.exists(full_path):
        try:
            # 1월 파일을 확인해보니 첫 행이 헤더이므로 skiprows는 제거했습니다.
            # 한글 깨짐 방지를 위해 cp949 인코딩을 우선 적용합니다.
            df = pd.read_csv(full_path, encoding='cp949')
            towing_list.append(df)
            print(f"{i}월 파일 병합 성공! (데이터 수: {len(df)}개)")
        except Exception as e:
            # 혹시 에러가 나면 utf-8로 다시 시도
            df = pd.read_csv(full_path, encoding='utf-8')
            towing_list.append(df)
            print(f"{i}월 파일 병합 성공! (utf-8 사용)")
    else:
        print(f"파일 없음: {file_name} - 경로를 확인해주세요.")

# 4. 최종 병합 및 저장
if towing_list:
    final_df = pd.concat(towing_list, ignore_index=True)
    print("\n" + "="*30)
    print(f"최종 병합 완료! 총 데이터 건수: {len(final_df)}건")
    
    # 분석용 파일로 저장 (Excel에서 볼 때 한글 안 깨지게 utf-8-sig 사용)
    output_path = os.path.join(path, '전체_견인_현황_2025.csv')
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"저장 위치: {output_path}")

1월 파일 병합 성공! (utf-8 사용)
2월 파일 병합 성공! (utf-8 사용)
3월 파일 병합 성공! (utf-8 사용)
4월 파일 병합 성공! (utf-8 사용)
5월 파일 병합 성공! (utf-8 사용)
6월 파일 병합 성공! (utf-8 사용)
7월 파일 병합 성공! (utf-8 사용)
8월 파일 병합 성공! (utf-8 사용)
9월 파일 병합 성공! (utf-8 사용)
10월 파일 병합 성공! (utf-8 사용)
11월 파일 병합 성공! (utf-8 사용)
12월 파일 병합 성공! (utf-8 사용)

최종 병합 완료! 총 데이터 건수: 64734건
저장 위치: C:\Users\chpar\Exercise_CP\work\전체_견인_현황_2025.csv


In [11]:
step 2 데이터 구조 확인(병합된 견인 데이터 & 주차구역 데이터)

import pandas as pd
import os

# 작업 경로 설정
path = r'C:\Users\chpar\Exercise_CP\work'

# 1. 병합된 전체 견인 데이터 로드
towing_df = pd.read_csv(os.path.join(path, '서울특별시_전동킥보드_전체_견인_현황_2025.csv'), encoding='utf-8-sig')

# 2. 주차구역 현황 데이터 로드
parking_df = pd.read_csv(os.path.join(path, '서울시 전동킥보드 주차구역 현황.csv'), encoding='cp949')

print("견인 데이터 컬럼:", towing_df.columns.tolist())
print("주차구역 데이터 컬럼:", parking_df.columns.tolist())

견인 데이터 컬럼: ['번호', '신고일', '구정보', '주소', '유형', '조치일']
주차구역 데이터 컬럼: ['순번', '시군구명', '주소', '상세위치', '거치대 유무']


In [3]:
# step 3 지오코당 진행(견인 데이터)

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time

# 1. 파일 로드
path = r'C:\Users\chpar\Exercise_CP\work'
file_name = '서울특별시_전동킥보드_전체_견인_현황_2025.csv'
df = pd.read_csv(os.path.join(path, file_name), encoding='utf-8-sig')

# 2. 좌표 컬럼이 없다면 생성 (이미 50개 성공했다면 있을 거야)
if 'lat' not in df.columns:
    df['lat'] = None
    df['lon'] = None

# 3. 지오코더 설정
geolocator = Nominatim(user_agent="ajou_pm_project_final")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2) # 속도를 조금 늦춰 안정성 확보

# 4. 전체 루프 실행
print("전체 지오코딩을 시작합니다. 이미 처리된 데이터는 건너뜁니다.")

count = 0
for idx, row in df.iterrows():
    # 'lat'이 비어 있는(NaN) 데이터만 처리
    if pd.isna(row['lat']):
        try:
            location = geocode(row['주소'])
            if location:
                df.at[idx, 'lat'] = location.latitude
                df.at[idx, 'lon'] = location.longitude
            
            count += 1
            
            # 10개마다 중간 저장 (혹시 모를 오류 대비)
            if count % 10 == 0:
                df.to_csv(os.path.join(path, 'towing_progress.csv'), index=False, encoding='utf-8-sig')
                print(f"{idx+1}번째 데이터 처리 중... (중간 저장 완료)")
                
        except Exception as e:
            print(f"{idx+1}번 행에서 오류 발생: {e}")
            time.sleep(2) # 오류 발생 시 잠시 휴식
            continue

# 5. 최종 결과 저장
df.to_csv(os.path.join(path, 'towing_final_geocoded.csv'), index=False, encoding='utf-8-sig')
print("--- 전체 지오코딩 완료 및 최종 저장 성공! ---")

전체 지오코딩을 시작합니다. 이미 처리된 데이터는 건너뜁니다.
10번째 데이터 처리 중... (중간 저장 완료)
20번째 데이터 처리 중... (중간 저장 완료)
30번째 데이터 처리 중... (중간 저장 완료)
40번째 데이터 처리 중... (중간 저장 완료)
50번째 데이터 처리 중... (중간 저장 완료)
60번째 데이터 처리 중... (중간 저장 완료)
70번째 데이터 처리 중... (중간 저장 완료)
80번째 데이터 처리 중... (중간 저장 완료)
90번째 데이터 처리 중... (중간 저장 완료)
100번째 데이터 처리 중... (중간 저장 완료)
110번째 데이터 처리 중... (중간 저장 완료)
120번째 데이터 처리 중... (중간 저장 완료)
130번째 데이터 처리 중... (중간 저장 완료)
140번째 데이터 처리 중... (중간 저장 완료)
150번째 데이터 처리 중... (중간 저장 완료)
160번째 데이터 처리 중... (중간 저장 완료)
170번째 데이터 처리 중... (중간 저장 완료)
180번째 데이터 처리 중... (중간 저장 완료)
190번째 데이터 처리 중... (중간 저장 완료)
200번째 데이터 처리 중... (중간 저장 완료)
210번째 데이터 처리 중... (중간 저장 완료)
220번째 데이터 처리 중... (중간 저장 완료)
230번째 데이터 처리 중... (중간 저장 완료)
240번째 데이터 처리 중... (중간 저장 완료)
250번째 데이터 처리 중... (중간 저장 완료)
260번째 데이터 처리 중... (중간 저장 완료)
270번째 데이터 처리 중... (중간 저장 완료)
280번째 데이터 처리 중... (중간 저장 완료)
290번째 데이터 처리 중... (중간 저장 완료)
300번째 데이터 처리 중... (중간 저장 완료)
310번째 데이터 처리 중... (중간 저장 완료)
320번째 데이터 처리 중... (중간 저장 완료)
330번째 데이터 처리 중... (중간 저장 완료)
340번째 데이터 처리 중...

In [5]:
# step 4 지오코당 진행(주차구역 데이터)

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time

# 1. 파일 로드 및 경로 설정
path = r'C:\Users\chpar\Exercise_CP\work'
parking_file = '서울시 전동킥보드 주차구역 현황.csv'

# 주차구역 데이터는 보통 cp949 인코딩입니다.
parking_df = pd.read_csv(os.path.join(path, parking_file), encoding='cp949')

# 2. 좌표 컬럼 초기화
if 'lat' not in parking_df.columns:
    parking_df['lat'] = None
    parking_df['lon'] = None

# 3. 지오코더 설정 (안정성을 위해 1.5초 간격 권장)
geolocator = Nominatim(user_agent="ajou_pm_parking_supply")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# 4. 주소 데이터 전처리 및 지오코딩 실행
print("주차구역 현황 지오코딩을 시작합니다...")

count = 0
for idx, row in parking_df.iterrows():
    # 이미 좌표가 있는 행은 건너뜁니다.
    if pd.isna(row['lat']):
        try:
            # '주소' 컬럼을 사용합니다.
            location = geocode(row['주소'])
            if location:
                parking_df.at[idx, 'lat'] = location.latitude
                parking_df.at[idx, 'lon'] = location.longitude
            
            count += 1
            
            # 10개마다 중간 저장
            if count % 10 == 0:
                parking_df.to_csv(os.path.join(path, 'parking_progress.csv'), index=False, encoding='utf-8-sig')
                print(f"{idx+1}번째 주차구역 처리 중...")
                
        except Exception as e:
            print(f"오류 발생 ({idx+1}번 행): {e}")
            time.sleep(2)
            continue

# 5. 최종 결과 저장
parking_df.to_csv(os.path.join(path, 'parking_final_geocoded.csv'), index=False, encoding='utf-8-sig')
print("--- 주차구역 지오코딩 완료 및 저장 성공! ---")

주차구역 현황 지오코딩을 시작합니다...
10번째 주차구역 처리 중...
20번째 주차구역 처리 중...
30번째 주차구역 처리 중...
40번째 주차구역 처리 중...
50번째 주차구역 처리 중...
60번째 주차구역 처리 중...
70번째 주차구역 처리 중...
80번째 주차구역 처리 중...
90번째 주차구역 처리 중...
100번째 주차구역 처리 중...
110번째 주차구역 처리 중...
120번째 주차구역 처리 중...
130번째 주차구역 처리 중...
140번째 주차구역 처리 중...
150번째 주차구역 처리 중...
160번째 주차구역 처리 중...
170번째 주차구역 처리 중...
180번째 주차구역 처리 중...
190번째 주차구역 처리 중...
200번째 주차구역 처리 중...
210번째 주차구역 처리 중...
220번째 주차구역 처리 중...
230번째 주차구역 처리 중...
240번째 주차구역 처리 중...
250번째 주차구역 처리 중...
260번째 주차구역 처리 중...
270번째 주차구역 처리 중...
280번째 주차구역 처리 중...
290번째 주차구역 처리 중...
300번째 주차구역 처리 중...
310번째 주차구역 처리 중...
320번째 주차구역 처리 중...
--- 주차구역 지오코딩 완료 및 저장 성공! ---


In [9]:
pip install geopy scikit-learn folium tqdm

Note: you may need to restart the kernel to use updated packages.


In [5]:
# step 4 지오코당 진행(주차구역 데이터)

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time

# 1. 파일 로드 및 경로 설정
path = r'C:\Users\chpar\Exercise_CP\work'
file_name = 'parking_final_geocoded.csv' # 수정하신 파일명

# 파일 읽기 (utf-8-sig는 한글 깨짐 방지용)
parking_df = pd.read_csv(os.path.join(path, file_name), encoding='utf-8-sig')

# 2. 지오코더 설정
# 안정적인 연결을 위해 user_agent를 고유하게 설정하고 간격을 1.5초로 유지합니다.
geolocator = Nominatim(user_agent="ajou_pm_parking_retry_v2")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# 3. 재지오코딩 실행 (lat이 비어 있는 행만 필터링)
print("누락된 주차구역 좌표를 다시 찾습니다...")

retry_count = 0
for idx, row in parking_df.iterrows():
    # 'lat' 컬럼이 비어 있는(NaN) 경우에만 실행
    if pd.isna(row['lat']):
        try:
            # 수정하신 '주소' 컬럼을 사용하여 검색
            location = geocode(row['주소'])
            
            if location:
                parking_df.at[idx, 'lat'] = location.latitude
                parking_df.at[idx, 'lon'] = location.longitude
                print(f"[성공] {idx+1}번 행: {row['주소']}")
                retry_count += 1
            else:
                print(f"[실패] {idx+1}번 행: {row['주소']} (검색 결과 없음)")
            
            # 5개 성공할 때마다 중간 저장 (데이터 보호)
            if retry_count % 5 == 0 and retry_count > 0:
                parking_df.to_csv(os.path.join(path, 'parking_final_geocoded_updated.csv'), index=False, encoding='utf-8-sig')
                
        except Exception as e:
            print(f"[오류] {idx+1}번 행: {e}")
            time.sleep(2)
            continue

# 4. 최종 결과 저장
final_output = os.path.join(path, 'parking_final_geocoded_updated.csv')
parking_df.to_csv(final_output, index=False, encoding='utf-8-sig')

print("\n" + "="*30)
print(f"재지오코딩 완료! 추가 성공 건수: {retry_count}건")
print(f"최종 파일 저장 위치: {final_output}")
print("="*30)

누락된 주차구역 좌표를 다시 찾습니다...
[실패] 1번 행: 팔판동 115-63  (검색 결과 없음)
[성공] 4번 행: 동숭동 1-24
[실패] 9번 행: 종로구 신문로 2가 58 (검색 결과 없음)
[실패] 19번 행: 중구 을지로 2가 206 (검색 결과 없음)
[실패] 27번 행: 성동구 성수동1가 656-302 (검색 결과 없음)
[실패] 31번 행: 성동구 행당동 산17 (검색 결과 없음)
[실패] 43번 행: 전농동 103-340 (검색 결과 없음)
[실패] 49번 행: 성북구 월곡동 37-4 (검색 결과 없음)
[성공] 60번 행: 창동 723-11
[성공] 63번 행: 화랑로 510
[성공] 64번 행: 공릉로 232
[성공] 65번 행: 공릉로 232
[성공] 69번 행: 동일로 1308-1
[실패] 72번 행: 동일로1530-1 (검색 결과 없음)
[성공] 75번 행: 동일로 1530-1
[성공] 76번 행: 노해로 502
[성공] 77번 행: 동일로 1308-1
[성공] 78번 행: 상계로 301-1
[성공] 82번 행: 동일로 1074
[성공] 83번 행: 동일로 1074
[성공] 85번 행: 동일로 1196
[성공] 86번 행: 동일로 1196
[성공] 88번 행: 동일로 1456
[성공] 100번 행: 신촌로 93
[성공] 105번 행: 공덕동 255-10
[성공] 112번 행: 서교동 490
[성공] 113번 행: 서교동 490
[실패] 123번 행: 마곡동 728-163 (검색 결과 없음)
[실패] 129번 행: 마곡동 728-190 (검색 결과 없음)
[실패] 141번 행: 공항대로 지하 163 (검색 결과 없음)
[성공] 194번 행: 서초동 711-4
[실패] 195번 행: 서초동 산 150-124 (검색 결과 없음)
[실패] 196번 행: 서초동 산 150-124 (검색 결과 없음)
[실패] 197번 행: 서초동 산 150-124 (검색 결과 없음)
[성공] 199번 행: 서초동 1445-17
[성공] 200번 행: 서초

In [7]:
# step 5 DBSCAN & 최단거리 계산 & 시각화

import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN
from geopy.distance import great_circle
import folium
from folium.plugins import HeatMap

# 1. 파일 로드 및 경로 설정
path = r'C:\Users\chpar\Exercise_CP\work'
towing_file = 'towing_final_geocoded.csv'
parking_file = 'parking_final_geocoded_updated.csv' # 수정된 파일 반영

# 데이터 읽기
towing_df = pd.read_csv(os.path.join(path, towing_file), encoding='utf-8-sig')
parking_df = pd.read_csv(os.path.join(path, parking_file), encoding='utf-8-sig')

# --- 데이터 정제: 좌표가 없는 행 제거 ---
towing_df = towing_df.dropna(subset=['lat', 'lon'])
parking_df = parking_df.dropna(subset=['lat', 'lon'])

# 2. DBSCAN 클러스터링 (수요 핫스팟 도출)
# 기준: 반경 30m(epsilon), 최소 10건 이상(min_samples)
coords = np.radians(towing_df[['lat', 'lon']])
epsilon = 0.03 / 6371  # 30m를 라디안으로 환산
db = DBSCAN(eps=epsilon, min_samples=10, algorithm='ball_tree', metric='haversine').fit(coords)
towing_df['cluster'] = db.labels_

# 노이즈(-1)를 제외한 클러스터별 중심점(핫스팟) 계산
hotspots = towing_df[towing_df['cluster'] != -1].groupby('cluster')[['lat', 'lon']].mean().reset_index()

# 3. 최단 거리 계산 (Mismatch Analysis)
# 각 핫스팟 중심에서 가장 가까운 기존 주차장까지의 거리(m) 산출
def get_nearest_dist(hotspot_point, parking_points):
    # great_circle을 사용하여 실제 지표면 거리를 계산
    min_d = min([great_circle(hotspot_point, p).meters for p in parking_points])
    return min_d

parking_coords = parking_df[['lat', 'lon']].values
print("미스매치 거리 분석 중...")
hotspots['dist_to_parking'] = hotspots.apply(lambda x: get_nearest_dist((x['lat'], x['lon']), parking_coords), axis=1)

# 4. 최종 시각화
# 지도의 중심은 데이터의 평균 위치로 설정
m = folium.Map(location=[towing_df['lat'].mean(), towing_df['lon'].mean()], zoom_start=14)

# (A) 견인 밀집도 히트맵
HeatMap(towing_df[['lat', 'lon']]).add_to(m)

# (B) 보완된 공식 주차구역 표시 (파란색 작은 원)
for _, row in parking_df.iterrows():
    folium.CircleMarker([row['lat'], row['lon']], radius=2, color='blue', fill=True, opacity=0.4).add_to(m)

# (C) 신규 주차구역 추천 지점 (거리가 200m 이상인 '사각지대' 핫스팟만 빨간색 별로 표시)
urgent_targets = hotspots[hotspots['dist_to_parking'] >= 200]
for _, row in urgent_targets.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=f"신설시급(거리:{row['dist_to_parking']:.1f}m)",
        icon=folium.Icon(color='red', icon='star')
    ).add_to(m)

# 5. 결과 저장 및 출력
m.save(os.path.join(path, 'final_transport_analysis_map.html'))
hotspots.to_csv(os.path.join(path, 'final_hotspot_analysis_results.csv'), index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"분석 결과 요약:")
print(f"- 추출된 총 수요 핫스팟: {len(hotspots)}개")
print(f"- 인프라 부족(200m 초과) 지점: {len(urgent_targets)}개")
print(f"- 핫스팟의 평균 주차장 접근 거리: {hotspots['dist_to_parking'].mean():.2f}m")
print(f"- 결과 저장 완료: final_transport_analysis_map.html / csv")
print("="*50)

미스매치 거리 분석 중...

분석 결과 요약:
- 추출된 총 수요 핫스팟: 213개
- 인프라 부족(200m 초과) 지점: 153개
- 핫스팟의 평균 주차장 접근 거리: 943.15m
- 결과 저장 완료: final_transport_analysis_map.html / csv
